# Greedy Agent Baseline
Each agent independently navigates toward the nearest shelf (if empty-handed) or nearest packing station (if carrying). No shared training, no communication between agents.

In [7]:
import sys
sys.path.insert(0, '..')

import yaml
import numpy as np
from src.env.warehouse_env import WarehouseEnv
from src.analytics import RewardTracker

with open('../configs/env_config.yaml') as f:
    config = yaml.safe_load(f)

env = WarehouseEnv(config)
print(f"Env: {config['env']['name']}  |  Agents: {env.n_agents}  |  Action dim: {env.action_dim}")

Env: rware-tiny-2ag-v2  |  Agents: 2  |  Action dim: 5


In [ ]:
# rware actions
NOOP      = 0
FORWARD   = 1
LEFT      = 2
RIGHT     = 3
INTERACT  = 4

# rware Direction enum: UP=0, DOWN=1, LEFT=2, RIGHT=3
DIR_TO_DELTA = {0: (0, -1), 1: (0, 1), 2: (-1, 0), 3: (1, 0)}
LEFT_RESULT  = {0: 2, 1: 3, 2: 1, 3: 0}

def nearest(pos, targets):
    ax, ay = pos
    return min(targets, key=lambda t: abs(t[0] - ax) + abs(t[1] - ay))

def desired_dir(ax, ay, tx, ty):
    dx, dy = tx - ax, ty - ay
    if dx == 0 and dy == 0: return None
    if abs(dx) >= abs(dy): return 3 if dx > 0 else 2
    else: return 1 if dy > 0 else 0

def forward_cell(agent):
    dx, dy = DIR_TO_DELTA[agent.dir.value]
    return (agent.x + dx, agent.y + dy)

def is_blocked_by_agent(agent, all_agents):
    nxt = forward_cell(agent)
    return any(a is not agent and (a.x, a.y) == nxt for a in all_agents)

def move_toward(agent, target, all_agents):
    ax, ay = agent.x, agent.y
    tx, ty = target
    if ax == tx and ay == ty:
        return INTERACT
    want = desired_dir(ax, ay, tx, ty)
    cur  = agent.dir.value
    if cur == want:
        if is_blocked_by_agent(agent, all_agents):
            return LEFT if np.random.rand() < 0.5 else RIGHT
        return FORWARD
    if LEFT_RESULT[cur] == want:
        return LEFT
    return RIGHT

def greedy_policy(env):
    u = env._env.unwrapped
    goal_list = list(u.goals)   # each goal is (x, y) in agent coords

    # Only target REQUESTED shelves (these give reward on delivery)
    requested = [(s.x, s.y) for s in u.request_queue]

    # Requested shelves not currently carried
    carried_ids = {id(a.carrying_shelf) for a in u.agents if a.carrying_shelf}
    available   = [(s.x, s.y) for s in u.request_queue if id(s) not in carried_ids]

    actions = []
    for agent in u.agents:
        pos = (agent.x, agent.y)

        if agent.carrying_shelf:
            # Carrying a requested shelf → head to nearest goal and drop
            target = nearest(pos, goal_list)
            actions.append(move_toward(agent, target, u.agents))

        else:
            # Not carrying → go pick up nearest requested shelf
            if not available:
                actions.append(NOOP)
                continue
            target = nearest(pos, available)
            actions.append(move_toward(agent, target, u.agents))

    return actions

In [10]:
N_EPISODES = 10000
max_steps  = config['env'].get('max_steps', 500)
tracker    = RewardTracker(n_agents=env.n_agents)

for ep in range(N_EPISODES):
    env.reset()
    tracker.start_episode()
    for _ in range(max_steps):
        actions = greedy_policy(env)
        _, rews, dones, _ = env.step(actions)
        tracker.record_step(rews)
        if all(dones):
            break
    tracker.end_episode()
    ep_data = tracker.episodes[-1]
    if (ep + 1) % 500 == 0:
        print(f"  ep {ep+1:5d}/{N_EPISODES}  team_reward={ep_data['team_total_reward']:.3f}")

env.close()

  ep   500/10000  team_reward=0.000
  ep  1000/10000  team_reward=0.000
  ep  1500/10000  team_reward=0.000
  ep  2000/10000  team_reward=0.000
  ep  2500/10000  team_reward=0.000
  ep  3000/10000  team_reward=0.000


KeyboardInterrupt: 

In [4]:
summary = tracker.summary()
print('--- Greedy Baseline Summary ---')
print(f"  Episodes : {summary['n_episodes']}")
print(f"  Team total reward  — mean: {summary['team_total_reward']['mean']:.4f}  "
      f"std: {summary['team_total_reward']['std']:.4f}  "
      f"[{summary['team_total_reward']['min']:.4f}, {summary['team_total_reward']['max']:.4f}]")
print(f"  Episode length     — mean: {summary['episode_length']['mean']:.1f}")
for i, v in enumerate(summary['per_agent_mean_total']):
    print(f"  Agent {i} mean total reward: {v:.4f}")

--- Greedy Baseline Summary ---
  Episodes : 10000
  Team total reward  — mean: 0.0000  std: 0.0000  [0.0000, 0.0000]
  Episode length     — mean: 500.0
  Agent 0 mean total reward: 0.0000
  Agent 1 mean total reward: 0.0000


In [5]:
import matplotlib.pyplot as plt

rewards = [ep['team_total_reward'] for ep in tracker.episodes]
plt.figure(figsize=(10, 4))
plt.bar(range(len(rewards)), rewards, color='seagreen', alpha=0.6)
plt.axhline(summary['team_total_reward']['mean'], color='red', linestyle='--',
            label=f"mean={summary['team_total_reward']['mean']:.3f}")
plt.xlabel('Episode')
plt.ylabel('Team Total Reward')
plt.title('Greedy Agent Baseline — Reward per Episode')
plt.legend()
plt.tight_layout()
plt.savefig('../results/plots/greedy_baseline_reward.png', dpi=150)
plt.show()

/var/folders/gb/c0f1t3sn061g3305zd78xrj40000gn/T/ipykernel_19699/1682842413.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
tracker.save('../results/logs/greedy_baseline_rewards.json')
tracker.save_csv('../results/logs/greedy_baseline_rewards.csv')
print('Saved to results/logs/')

[analytics] Saved reward log → ../results/logs/greedy_baseline_rewards.json
[analytics] Saved CSV → ../results/logs/greedy_baseline_rewards.csv
Saved to results/logs/
